# Data augmentation

_Deep Learning_

**Make more training images by flipping, rotating, and cropping the ones you have.**

More data usually means a better model. But collecting it is expensive.

     Data augmentation creates new training examples from existing ones with simple changes: flip, rotate, crop, shift colors.

---

This notebook is a practice scaffold for the **Data augmentation** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
from torchvision import transforms

augment = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),     # mirror left-right
    transforms.RandomRotation(degrees=15),      # tilt a little
    transforms.RandomResizedCrop(32, scale=(0.8, 1.0)),  # zoom/crop
    transforms.ColorJitter(brightness=0.2),     # tweak brightness
])

# a fake 3-channel 40x40 image as a tensor
img = torch.rand(3, 40, 40)

# each call yields a different random variant of the same image
v1 = augment(img)
v2 = augment(img)
print("augmented shape:", tuple(v1.shape))      # (3, 32, 32)
print("two variants differ:", not torch.equal(v1, v2))

## Visualize the data & results

_On real digits seen at an angle, does training-time augmentation actually improve accuracy?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from scipy.ndimage import rotate, shift

digits = load_digits()
Xtr, Xte, ytr, yte = train_test_split(digits.images, digits.target,
                                      test_size=0.3, random_state=0,
                                      stratify=digits.target)
Xte_rot = np.array([rotate(im, 18, reshape=False) for im in Xte])  # test digits at an angle

def accuracy(augs):
    imgs, labs = [Xtr], [ytr]
    ops = {"rot": 15, "rotn": -15, "rot2": 20}
    for a in augs:
        if a in ops:
            imgs.append(np.array([rotate(im, ops[a], reshape=False) for im in Xtr]))
        else:
            imgs.append(np.array([shift(im, [1, 1]) for im in Xtr]))
        labs.append(ytr)
    X = np.vstack([i.reshape(len(i), -1) for i in imgs]) / 16.0
    Y = np.concatenate(labs)
    m = MLPClassifier(hidden_layer_sizes=(50,), max_iter=400,
                      random_state=0, alpha=1e-3).fit(X, Y)
    return accuracy_score(yte, m.predict(Xte_rot.reshape(len(Xte_rot), -1) / 16.0))

levels = [[], ["rot"], ["rot", "rotn"], ["rot", "rotn", "rot2"],
          ["rot", "rotn", "rot2", "shift"]]
labels = ["no aug", "+rotate15", "+rotate-15", "+rotate20", "+shift (all)"]
accs = [accuracy(a) for a in levels]
colors = ["#ff7b72", "#ffb454", "#ffb454", "#4ea1ff", "#7ee787"]

plt.bar(labels, accs, color=colors)
plt.title("Real accuracy on rotated test digits as augmentations are added")
plt.ylim(0.6, 1.0)
plt.show()